 ### Imports & Configuration
On charge les bibliothèques Python nécessaires.
 - `pandas` : lire, manipuler et sauvegarder les CSV
 - `os` : gérer les chemins de fichiers
 - `re` : manipuler les chaînes de texte

In [3]:
import pandas as pd
import os
import re

# ── Chemins ──────────────────────────────────────────────────────────────────
# Modifiez INPUT_DIR pour pointer vers le dossier où sont vos CSV
INPUT_DIR  = "./tables"   # dossier contenant les CSV bruts Airtable
OUTPUT_DIR = "./clean_tables"      # dossier où seront sauvegardés les CSV nettoyés

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(" Configuration OK")
print(f"   Lecture depuis  : {os.path.abspath(INPUT_DIR)}")
print(f"   Écriture vers   : {os.path.abspath(OUTPUT_DIR)}")

 Configuration OK
   Lecture depuis  : c:\Maria\digitalPlanner\tables
   Écriture vers   : c:\Maria\digitalPlanner\clean_tables


## Chargement de tous les CSV
- On charge tous les fichiers en mémoire d'un coup pour pouvoir les comparer entre eux.

- le  paramètre `encoding='utf-8-sig'` gère le BOM (caractère invisible en début de fichier que Windows/Excel ajoute parfois).

In [4]:
# Chargement de chaque table
df_availability   = pd.read_csv(f"{INPUT_DIR}/MD_Availability-Grid view.csv",     encoding="utf-8-sig")
df_customer       = pd.read_csv(f"{INPUT_DIR}/MD_Customer-Grid view.csv",          encoding="utf-8-sig")
df_holidays       = pd.read_csv(f"{INPUT_DIR}/MD_Holidays-Grid view.csv",          encoding="utf-8-sig")
df_operations     = pd.read_csv(f"{INPUT_DIR}/MD_Operation-Grid view.csv",         encoding="utf-8-sig")
df_skills         = pd.read_csv(f"{INPUT_DIR}/MD_Skills-Grid view.csv",            encoding="utf-8-sig")
df_technicians    = pd.read_csv(f"{INPUT_DIR}/MD_Technicians-Grid view.csv",       encoding="utf-8-sig")
df_work_centers   = pd.read_csv(f"{INPUT_DIR}/MD_Work Centers-Grid view.csv",      encoding="utf-8-sig")
df_woo            = pd.read_csv(f"{INPUT_DIR}/Work Order Operations-Grid view.csv",encoding="utf-8-sig")
df_wo             = pd.read_csv(f"{INPUT_DIR}/Work Order-Grid view.csv",            encoding="utf-8-sig")
df_asset_crit     = pd.read_csv(f"{INPUT_DIR}/MD_Asset Criticality-Grid view.csv", encoding="utf-8-sig")
df_assets         = pd.read_csv(f"{INPUT_DIR}/MD_Assets-Grid view.csv",            encoding="utf-8-sig")

# Affichage d'un résumé : nombre de lignes et colonnes par table
tables = {
    "Availability"        : df_availability,
    "Customer"            : df_customer,
    "Holidays"            : df_holidays,
    "Operations"          : df_operations,
    "Skills"              : df_skills,
    "Technicians"         : df_technicians,
    "Work_Centers"        : df_work_centers,
    "Work_Order_Ops"      : df_woo,
    "Work_Order"          : df_wo,
    "Asset_Criticality"   : df_asset_crit,
    "Assets"              : df_assets,
}

print(f"{'Table':<22} {'Lignes':>8} {'Colonnes':>10}")
print("-" * 42)
for name, df in tables.items():
    print(f"{name:<22} {len(df):>8} {len(df.columns):>10}")

Table                    Lignes   Colonnes
------------------------------------------
Availability                  2          7
Customer                     62          4
Holidays                      0          7
Operations                   27          8
Skills                        4          3
Technicians                   7         21
Work_Centers                  3          6
Work_Order_Ops             1096         21
Work_Order                    1         12
Asset_Criticality             3          2
Assets                      145         15


## Inspection visuelle des colonnes
- Avant de supprimer quoi que ce soit, on vérifie exactement quels noms de colonnes existent dans chaque table. Les noms dans Airtable peuvent différer légèrement de notre audit (espaces, majuscules, etc.).

In [5]:
for name, df in tables.items():
    print(f"\n{'='*10} {name} {'='*10}")
    for col in df.columns:
        print(f"   • {col}")


========== Availability ==========
   • Working Hour ID
   • Status
   • Work Hour Description
   • Start Hour
   • End Hour
   • Pause start
   • Pause end

========== Customer ==========
   • Customer ID
   • Status
   • Customer Description
   • MD_Assets

========== Holidays ==========
   • Holiday request day and time
   • Status
   • Requested by technician
   • Technician first and last name (from Requested by technician)
   • Requested for
   • Full or half day
   • Raison

========== Operations ==========
   • Operation ID
   • Operation key
   • Operation Description
   • Duration
   • Duration unit
   • MD_Skills_Possibilities
   • Work Order Operations
   • Predescesor Operation key

========== Skills ==========
   • Skill ID
   • Skill Description
   • MD_Technicians

========== Technicians ==========
   • Technician ID
   • Technicial status
   • Technician First Name
   • Technician Last Name
   • Technician first and last name
   • MD_Skills
   • Skill Description
   •

In [6]:
print(f"Nombre de colonnes : {len(df_customer.columns)}")
print(f"Colonnes : {list(df_customer.columns)}")

Nombre de colonnes : 4
Colonnes : ['Customer ID', 'Status', 'Customer Description', 'MD_Assets']


## Renommage des colonnes (standardisation)
- Airtable exporte des noms de colonnes avec des espaces, des parenthèses, des accents... 
- On les renomme en `snake_case` propre pour faciliter tout le reste du travail.

- Exemple : `"Technician First Name"` → `"technician_first_name"`

In [7]:
# ── Table : Availability ──────────────────────────────────────────────────────
# On garde : ID, status, description, start_hour, end_hour, pause_start, pause_end
df_availability.columns = [
    "pk_working_hour_id",
    "status",
    "working_hour_description",
    "start_hour",
    "end_hour",
    "pause_start",
    "pause_end"
]

# ── Table : Customer ─────────────────────────────────────────────────────────
# On supprime la colonne MD_Assets (liste d'IDs dénormalisée — la relation
# existe déjà dans Assets via fk_customer)
df_customer.columns = ["pk_customer_id", "status", "customer_description", "_assets_ids"]
df_customer = df_customer.drop(columns=["_assets_ids"])

# ── Table : Holidays ─────────────────────────────────────────────────────────
# Problème structurel : la FK pointe sur un NOM au lieu d'un ID → on va corriger ça
if len(df_holidays) > 0:
    df_holidays.columns = [
        "holiday_request_time",
        "status",
        "fk_technician_id",                  # sera l'ID après correction
        "technician_name_raw",               # nom brut — sera supprimé après jointure
        "requested_for",
        "full_or_half_day",
        "reason"
    ]
else:
    print(" Holidays est vide — création d'un DataFrame vide avec le bon schéma")
    df_holidays = pd.DataFrame(columns=[
        "holiday_request_time", "status", "fk_technician_id",
        "requested_for", "full_or_half_day", "reason"
    ])

# ── Table : Operations ───────────────────────────────────────────────────────
# On supprime les colonnes de liaison dénormalisées (MD_Skills_Possibilities,
# Work Order Operations) — ces relations existent via FK dans d'autres tables
df_operations.columns = [
    "pk_operation_id",
    "operation_key",
    "operation_description",
    "duration",
    "duration_unit",
    "_skills_ids",          # liste dénormalisée → supprimée
    "_woo_ids",             # liste dénormalisée → supprimée
    "predecessor_operation_key"
]
df_operations = df_operations.drop(columns=["_skills_ids", "_woo_ids"])

# ── Table : Skills ───────────────────────────────────────────────────────────
# On supprime MD_Technicians (liste dénormalisée)
df_skills.columns = ["pk_skill_id", "skill_description", "_technicians_ids"]
df_skills = df_skills.drop(columns=["_technicians_ids"])

# ── Table : Technicians ──────────────────────────────────────────────────────
# NETTOYAGE MAJEUR :
# - Supprimer skill_description       (dupliqué depuis Skills)
# - Supprimer tous les champs availability (dupliqués depuis Availability)
# - Supprimer work_center_description (dupliqué depuis Work_Centers)
# - Conserver uniquement les FK vers les autres tables
df_technicians.columns = [
    "pk_technician_id",
    "technician_status",
    "technician_first_name",
    "technician_last_name",
    "technician_full_name",
    "fk_skill_id",
    "_skill_description_DUPLICATE",      # ← dupliqué de Skills → SUPPRIMÉ
    "fk_availability_id",
    "_availability_desc_DUPLICATE",      # ← dupliqué de Availability → SUPPRIMÉ
    "_start_hour_DUPLICATE",             # ← dupliqué de Availability → SUPPRIMÉ
    "_start_hour2_DUPLICATE",            # ← dupliqué de Availability → SUPPRIMÉ
    "_pause_start_DUPLICATE",            # ← dupliqué de Availability → SUPPRIMÉ
    "_pause_end_DUPLICATE",              # ← dupliqué de Availability → SUPPRIMÉ
    "address_street",
    "address_door",
    "address_post_code",
    "address_city",
    "fk_work_center_id",
    "_work_center_desc_DUPLICATE",       # ← dupliqué de Work_Centers → SUPPRIMÉ
    "_woo_ids",                          # liste dénormalisée → SUPPRIMÉE
    "_holidays_ids"                      # liste dénormalisée → SUPPRIMÉE
]
cols_to_drop_tech = [
    "_skill_description_DUPLICATE",
    "_availability_desc_DUPLICATE",
    "_start_hour_DUPLICATE",
    "_start_hour2_DUPLICATE",
    "_pause_start_DUPLICATE",
    "_pause_end_DUPLICATE",
    "_work_center_desc_DUPLICATE",
    "_woo_ids",
    "_holidays_ids"
]
df_technicians = df_technicians.drop(columns=cols_to_drop_tech)

# ── Table : Work_Centers ─────────────────────────────────────────────────────
# On supprime les colonnes dénormalisées (noms et compétences des techniciens)
# Ces infos sont disponibles via la FK fk_work_center_id dans Technicians
df_work_centers.columns = [
    "pk_work_center_id",
    "work_center_status",
    "work_center_description",
    "_technicians_ids",                  # liste dénormalisée → SUPPRIMÉE
    "_technician_names_DUPLICATE",       # dupliqué de Technicians → SUPPRIMÉ
    "_skill_desc_DUPLICATE"              # dupliqué de Skills → SUPPRIMÉ
]
df_work_centers = df_work_centers.drop(columns=[
    "_technicians_ids", "_technician_names_DUPLICATE", "_skill_desc_DUPLICATE"
])

# ── Table : Asset_Criticality ────────────────────────────────────────────────
# On supprime la liste d'IDs d'assets (relation inverse dénormalisée)
df_asset_crit.columns = ["pk_criticality_name", "_assets_ids"]
df_asset_crit = df_asset_crit.drop(columns=["_assets_ids"])

# ── Table : Assets ───────────────────────────────────────────────────────────
# NETTOYAGE MAJEUR :
# - Supprimer work_center_description (dupliqué de Work_Centers)
# - Supprimer customer_description    (dupliqué de Customer)
# - Supprimer Work Order              (liste dénormalisée)
df_assets.columns = [
    "pk_asset_id",
    "asset_status",
    "asset_description",
    "address_street",
    "address_door",
    "address_post_code",
    "address_city",
    "asset_region",
    "fk_work_center_id",
    "_work_center_desc_DUPLICATE",       # dupliqué de Work_Centers → SUPPRIMÉ
    "fk_customer_id",
    "_customer_desc_DUPLICATE",          # dupliqué de Customer → SUPPRIMÉ
    "_work_order_ids",                   # liste dénormalisée → SUPPRIMÉE
    "fk_criticality",
    "created_at"
]
df_assets = df_assets.drop(columns=[
    "_work_center_desc_DUPLICATE",
    "_customer_desc_DUPLICATE",
    "_work_order_ids"
])

# ── Table : Work_Order ───────────────────────────────────────────────────────
# NETTOYAGE MAJEUR :
# - Supprimer asset_description       (dupliqué de Assets)
# - Supprimer work_center_description (dupliqué de Work_Centers)
# - Supprimer customer_description    (dupliqué de Customer)
df_wo.columns = [
    "pk_work_order_id",
    "work_order_type",
    "_woo_ids",                          # liste dénormalisée → SUPPRIMÉE
    "breakdown",
    "priority",
    "order_basic_start_date",
    "order_basic_end_date",
    "last_inspection_date",
    "fk_asset_id",
    "_asset_desc_DUPLICATE",             # dupliqué de Assets → SUPPRIMÉ
    "_work_center_desc_DUPLICATE",       # dupliqué de Work_Centers → SUPPRIMÉ
    "_customer_desc_DUPLICATE"           # dupliqué de Customer → SUPPRIMÉ
]
df_wo = df_wo.drop(columns=[
    "_woo_ids",
    "_asset_desc_DUPLICATE",
    "_work_center_desc_DUPLICATE",
    "_customer_desc_DUPLICATE"
])

# ── Table : Work_Order_Operations ────────────────────────────────────────────
# NETTOYAGE MAJEUR :
# - Supprimer technician_first_name   (dupliqué de Technicians)
# - Supprimer asset_work_center       (dupliqué de Work_Centers via Assets)
# - Supprimer work_order_id_auto      (doublon de work_order_id)
df_woo.columns = [
    "pk_woo_id",
    "fk_work_order_id",
    "_work_order_id_auto_DUPLICATE",     # doublon → SUPPRIMÉ
    "status",
    "fk_operation_id",
    "operation_key",
    "operation_description",
    "duration",
    "duration_unit",
    "predecessor_operation_key",
    "fk_required_skill_id",
    "_required_skill_desc",              # utile pour lisibilité — on garde
    "order_basic_start_date",
    "order_basic_end_date",
    "operation_scheduled_start",
    "operation_scheduled_end",
    "confirmed_work",
    "confirmed_work_unit",
    "_asset_work_center_DUPLICATE",      # dupliqué via Assets → SUPPRIMÉ
    "fk_assigned_technician_id",
    "_technician_name_DUPLICATE"         # dupliqué de Technicians → SUPPRIMÉ
]
df_woo = df_woo.drop(columns=[
    "_work_order_id_auto_DUPLICATE",
    "_asset_work_center_DUPLICATE",
    "_technician_name_DUPLICATE"
])

print(" Renommage et suppression des colonnes dupliquées terminés")

 Holidays est vide — création d'un DataFrame vide avec le bon schéma
 Renommage et suppression des colonnes dupliquées terminés


##  Correction du problème structurel : Holidays
- Dans la table Holidays, la FK vers le technicien utilise un **nom** (`Luc Dupuis`) au lieu d'un **ID** (`6`). C'est dangereux : si un technicien change de nom, le lien est cassé.

- On corrige en faisant une **jointure** : on cherche l'ID correspondant au nom dans la table Technicians.

In [8]:
if len(df_holidays) > 0 and "technician_name_raw" in df_holidays.columns:
    # Créer un dictionnaire nom → ID depuis la table Technicians
    name_to_id = dict(zip(
        df_technicians["technician_full_name"],
        df_technicians["pk_technician_id"]
    ))
    
    # Remplacer le nom brut par l'ID correspondant
    df_holidays["fk_technician_id"] = df_holidays["technician_name_raw"].map(name_to_id)
    
    # Supprimer la colonne nom brut — elle ne sert plus
    df_holidays = df_holidays.drop(columns=["technician_name_raw"])
    
    # Vérifier qu'aucun ID n'est manquant (NaN = nom non trouvé dans Technicians)
    missing = df_holidays["fk_technician_id"].isna().sum()
    if missing > 0:
        print(f"  {missing} lignes Holidays sans technicien correspondant")
    else:
        print(" Holidays : FK corrigée — tous les noms ont un ID correspondant")
else:
    print("ℹ  Holidays est vide — rien à corriger")

ℹ  Holidays est vide — rien à corriger


## Découpage de operation_key en 3 champs distincts 

In [9]:

# Fonction qui parse "Corrective-Breakdown-1" → ('Corrective', 'Breakdown', 1)
def parse_operation_key(key):
    if pd.isna(key) or key == "":
        return None, None, None
    parts = str(key).split("-")
    op_type     = parts[0] if len(parts) > 0 else None   # Corrective / Preventive
    op_subtype  = parts[1] if len(parts) > 1 else None   # Breakdown / Low / Meca...
    op_order    = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else None
    return op_type, op_subtype, op_order

# Appliquer sur la table Operations
df_operations[["operation_type", "operation_subtype", "operation_order"]] = df_operations["operation_key"].apply(
    lambda k: pd.Series(parse_operation_key(k))
)

# Appliquer sur WOO aussi
df_woo[["operation_type", "operation_subtype", "operation_order"]] = df_woo["operation_key"].apply(
    lambda k: pd.Series(parse_operation_key(k))
)

# Vérification
print(df_operations[["operation_key", "operation_type", "operation_subtype", "operation_order"]].to_string())


              operation_key operation_type operation_subtype  operation_order
0    Corrective-Breakdown-1     Corrective         Breakdown                1
1    Corrective-Breakdown-2     Corrective         Breakdown                2
2    Corrective-Breakdown-3     Corrective         Breakdown                3
3    Corrective-Breakdown-4     Corrective         Breakdown                4
4    Corrective-Breakdown-5     Corrective         Breakdown                5
5    Corrective-Breakdown-6     Corrective         Breakdown                6
6    Corrective-Breakdown-7     Corrective         Breakdown                7
7    Corrective-Breakdown-8     Corrective         Breakdown                8
8    Corrective-Breakdown-9     Corrective         Breakdown                9
9   Corrective-Breakdown-10     Corrective         Breakdown               10
10         Corrective-Low-1     Corrective               Low                1
11         Corrective-Low-2     Corrective               Low    

- La table Work_Order n'a qu'1 enregistrement sur 177 attendus.
- toute l'info utile est déjà dans operation_type / operation_subtype / operation_order issus du découpage.

In [10]:
# Suppression de la table Work_Order — elle n'entrera pas dans la base SQLite
df_wo = None
print("Table Work_Order écartée — non intégrée dans la base SQLite")
print(f"\nColonnes finales de WOO : ")
for col in df_woo.columns:
    print(f"    {col}")

Table Work_Order écartée — non intégrée dans la base SQLite

Colonnes finales de WOO : 
    pk_woo_id
    fk_work_order_id
    status
    fk_operation_id
    operation_key
    operation_description
    duration
    duration_unit
    predecessor_operation_key
    fk_required_skill_id
    _required_skill_desc
    order_basic_start_date
    order_basic_end_date
    operation_scheduled_start
    operation_scheduled_end
    confirmed_work
    confirmed_work_unit
    fk_assigned_technician_id
    operation_type
    operation_subtype
    operation_order


In [11]:
df_woo = df_woo.rename(columns={"_required_skill_desc": "required_skill_desc"})


- Ajouter fk_asset_id dans WOO
- Après suppression de Work_Order, WOO ne sait plus quel asset est concerné par chaque opération.

- On a découvert que dans 143 cas sur 145 :  Work Order ID = Asset ID  (pattern direct)
- Pour les 34 cas restants (WO_ID > 146), on met NULL car ces assets ne sont pas identifiables sans info du client.
- COMMENT # On crée un mapping direct : si Work Order ID existe dans Assets → fk_asset_id = Work Order ID
- Sinon → fk_asset_id = None (NULL)
- On supprime maintenant fk_work_order_id 


In [12]:


asset_ids_valides = set(df_assets["pk_asset_id"].unique())

def get_asset_id(work_order_id):
    if work_order_id in asset_ids_valides:
        return work_order_id   # correspondance directe 
    return None                # orphelin → NULL 

df_woo["fk_asset_id"] = df_woo["fk_work_order_id"].apply(get_asset_id)

# Rapport
total        = len(df_woo)
avec_asset   = df_woo["fk_asset_id"].notna().sum()
sans_asset   = df_woo["fk_asset_id"].isna().sum()


print(f" Opérations avec asset identifié : {avec_asset} / {total}")
print(f"  Opérations sans asset (NULL)   : {sans_asset} / {total}")
print(f"\nExemple (5 premières lignes) :")
print(df_woo[["pk_woo_id", "fk_work_order_id", "fk_asset_id"]].head())

# On supprime maintenant fk_work_order_id — elle a servi, on n'en a plus besoin
df_woo = df_woo.drop(columns=["fk_work_order_id"])
print("\n fk_work_order_id supprimé de WOO")

 Opérations avec asset identifié : 983 / 1096
  Opérations sans asset (NULL)   : 113 / 1096

Exemple (5 premières lignes) :
   pk_woo_id  fk_work_order_id  fk_asset_id
0        132                 1          1.0
1        133                 1          1.0
2        134                 1          1.0
3        135                 1          1.0
4        136                 1          1.0

 fk_work_order_id supprimé de WOO


### Ajouter priority_score dans WOO
- L'algorithme de planification a besoin d'un NOMBRE pour trier les opérations par urgence.
- Il ne peut pas comparer du texte  comme "Breakdown" > "Low" directement.
- On traduit donc operation_subtype en score numérique :
- Breakdown = 5  ← panne totale, priorité maximale
- Meca      = 4  ← mécanique, urgent
- Elec      = 3  ← électrique, urgent
- monthly   = 2  ← préventif mensuel, planifiable
- Low       = 1  ← basse priorité, flexible

- On utilise un dictionnaire de mapping subtype → score Si le subtype est inconnu → score 0 (à investiguer)

In [13]:

priority_map = {
    "Breakdown" : 5,
    "Meca"      : 4,
    "Elec"      : 3,
    "monthly"   : 2,
    "Low"       : 1,
}

df_woo["priority_score"] = df_woo["operation_subtype"].map(priority_map).fillna(0).astype(int)

# Rapport
print(f"\nRépartition des scores :")
rapport = df_woo.groupby(["operation_subtype", "priority_score"]).size().reset_index(name="nb_operations")
print(rapport.to_string(index=False))

score_0 = df_woo[df_woo["priority_score"] == 0]
if len(score_0) > 0:
    print(f"\n  {len(score_0)} opérations avec score 0 (subtype inconnu) :")
    print(score_0[["pk_woo_id", "operation_subtype"]].head())
else:
    print(f"\n Tous les subtypes sont reconnus — aucun score 0")


Répartition des scores :
operation_subtype  priority_score  nb_operations
        Breakdown               5            190
             Elec               3             24
              Low               1            623
             Meca               4             39
          monthly               2            220

 Tous les subtypes sont reconnus — aucun score 0


### Vérification de l'intégrité des FK
- Avant de sauvegarder, on vérifie que toutes les FK pointent vers des IDs qui existent réellement dans leurs tables de référence.

- Exemple : chaque `fk_work_center_id` dans Technicians doit exister dans `pk_work_center_id` de Work_Centers.

In [14]:
def check_fk(child_df, child_col, parent_df, parent_col, label):
    """
    Vérifie qu'une FK ne pointe pas vers un ID inexistant.
    child_df[child_col]  → les valeurs FK à vérifier
    parent_df[parent_col] → les IDs de référence
    """
    # On ignore les valeurs nulles (FK optionnelles)
    child_vals  = set(child_df[child_col].dropna().astype(str))
    parent_vals = set(parent_df[parent_col].dropna().astype(str))
    orphans = child_vals - parent_vals
    if orphans:
        print(f" {label}: {len(orphans)} valeur(s) orpheline(s) → {list(orphans)[:5]}")
    else:
        print(f" {label}: OK")

print("── Vérification des Foreign Keys ───────────────────────────────")
check_fk(df_technicians, "fk_skill_id",         df_skills,       "pk_skill_id",         "Technicians → Skills")
check_fk(df_technicians, "fk_availability_id",  df_availability, "pk_working_hour_id",  "Technicians → Availability")
check_fk(df_technicians, "fk_work_center_id",   df_work_centers, "pk_work_center_id",   "Technicians → Work_Centers")
check_fk(df_assets,      "fk_work_center_id",   df_work_centers, "pk_work_center_id",   "Assets → Work_Centers")
check_fk(df_assets,      "fk_customer_id",      df_customer,     "pk_customer_id",      "Assets → Customer")
check_fk(df_assets,      "fk_criticality",      df_asset_crit,   "pk_criticality_name", "Assets → Asset_Criticality")
check_fk(df_woo,         "fk_operation_id",     df_operations,   "pk_operation_id",     "WOO → Operations")

── Vérification des Foreign Keys ───────────────────────────────
 Technicians → Skills: OK
 Technicians → Availability: OK
 Technicians → Work_Centers: OK
 Assets → Work_Centers: OK
 Assets → Customer: OK
 Assets → Asset_Criticality: OK
 WOO → Operations: OK


#### un résumé clair de ce qui a été fait : combien de colonnes supprimées par table, combien de lignes restantes.

In [15]:
tables_clean = {
    "Availability"      : df_availability,
    "Customer"          : df_customer,
    "Holidays"          : df_holidays,
    "Operations"        : df_operations,
    "Skills"            : df_skills,
    "Technicians"       : df_technicians,
    "Work_Centers"      : df_work_centers,
    "Work_Order_Ops"    : df_woo,
    "Asset_Criticality" : df_asset_crit,
    "Assets"            : df_assets,
}

print(f"{'Table':<22} {'Lignes':>8} {'Colonnes finales':>18}")
print("-" * 52)
for name, df in tables_clean.items():
    print(f"{name:<22} {len(df):>8} {len(df.columns):>18}")

print("\n── Colonnes finales par table ───────────────────────────────────")
for name, df in tables_clean.items():
    print(f"\n{name}:")
    for col in df.columns:
        print(f"   ✓ {col}")

Table                    Lignes   Colonnes finales
----------------------------------------------------
Availability                  2                  7
Customer                     62                  3
Holidays                      0                  6
Operations                   27                  9
Skills                        4                  2
Technicians                   7                 12
Work_Centers                  3                  3
Work_Order_Ops             1096                 22
Asset_Criticality             3                  1
Assets                      145                 12

── Colonnes finales par table ───────────────────────────────────

Availability:
   ✓ pk_working_hour_id
   ✓ status
   ✓ working_hour_description
   ✓ start_hour
   ✓ end_hour
   ✓ pause_start
   ✓ pause_end

Customer:
   ✓ pk_customer_id
   ✓ status
   ✓ customer_description

Holidays:
   ✓ holiday_request_time
   ✓ status
   ✓ fk_technician_id
   ✓ requested_for
   ✓ full_or_half

Sauvegarde des CSV propres
- On sauvegarde chaque table nettoyée dans un fichier CSV séparé
- dans le dossier clean_tables/ comme backup avant de créer la BD.
- Ces fichiers sont la version finale propre des données Airtable.

In [16]:


import os

export_map = {
    "availability.csv"           : df_availability,
    "customer.csv"               : df_customer,
    "holidays.csv"               : df_holidays,
    "operations.csv"             : df_operations,
    "skills.csv"                 : df_skills,
    "technicians.csv"            : df_technicians,
    "work_centers.csv"           : df_work_centers,
    "work_order_operations.csv"  : df_woo,
    "asset_criticality.csv"      : df_asset_crit,
    "assets.csv"                 : df_assets,
}

for filename, df in export_map.items():
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f" {filename:<35} ({len(df)} lignes, {len(df.columns)} colonnes)")

print(f"\n CSV propres sauvegardés dans : {os.path.abspath(OUTPUT_DIR)}")

 availability.csv                    (2 lignes, 7 colonnes)
 customer.csv                        (62 lignes, 3 colonnes)
 holidays.csv                        (0 lignes, 6 colonnes)
 operations.csv                      (27 lignes, 9 colonnes)
 skills.csv                          (4 lignes, 2 colonnes)
 technicians.csv                     (7 lignes, 12 colonnes)
 work_centers.csv                    (3 lignes, 3 colonnes)
 work_order_operations.csv           (1096 lignes, 22 colonnes)
 asset_criticality.csv               (3 lignes, 1 colonnes)
 assets.csv                          (145 lignes, 12 colonnes)

 CSV propres sauvegardés dans : c:\Maria\digitalPlanner\clean_tables


## CORRECTION FK avant import SQLite (table woo)


1 : fk_required_skill_id 
Problème : certaines cellules contiennent "1,2,4" au lieu de "1"
Solution : garder uniquement le premier skill de la liste

on garde le premier car C'est le skill principal requis
Les autres sont des alternatives — on les ignorera pour l'instant


2 fk_asset_id float → integer 
Problème : pandas a converti en float à cause des NULL (113 NULL)
Solution : convertir en Int64 (nullable integer de pandas)
Int64 avec majuscule = entier qui accepte les NULL (contrairement à int64)

3 fk_assigned_technician_id float → Int64 
Même problème que fk_asset_id — 1096 NULL (normal, pas encore assigné)
On convertit quand même pour cohérence avec le schéma SQLite

In [29]:


def extract_first_skill(val):
    if pd.isna(val) or val == "":
        return None
    # Si c'est une liste "1,2,4" → prendre "1"
    first = str(val).split(",")[0].strip()
    try:
        return int(first)
    except:
        return None

df_woo["fk_required_skill_id"] = df_woo["fk_required_skill_id"].apply(extract_first_skill)

print(" — fk_required_skill_id")
print(f"   Valeurs uniques : {sorted(df_woo['fk_required_skill_id'].dropna().unique().tolist())}")
print(f"   NULL restants   : {df_woo['fk_required_skill_id'].isna().sum()}")

#  fk_asset_id float → integer 


df_woo["fk_asset_id"] = pd.to_numeric(df_woo["fk_asset_id"], errors="coerce")
df_woo["fk_asset_id"] = df_woo["fk_asset_id"].astype("Int64")

print("\n _ fk_asset_id float → Int64")
print(f"   Type actuel     : {df_woo['fk_asset_id'].dtype}")
print(f"   NULL restants   : {df_woo['fk_asset_id'].isna().sum()}")
print(f"   Exemple valeurs : {df_woo['fk_asset_id'].dropna().head().tolist()}")

# fk_assigned_technician_id float → Int64 


df_woo["fk_assigned_technician_id"] = pd.to_numeric(
    df_woo["fk_assigned_technician_id"], errors="coerce"
)
df_woo["fk_assigned_technician_id"] = df_woo["fk_assigned_technician_id"].astype("Int64")

print("\n fk_assigned_technician_id float → Int64")
print(f"   Type actuel     : {df_woo['fk_assigned_technician_id'].dtype}")
print(f"   NULL restants   : {df_woo['fk_assigned_technician_id'].isna().sum()} (normal — pas encore assigné)")

# ── Vérification finale 
print("\n── Vérification après corrections ")
sk_ids     = set(df_skills["pk_skill_id"].dropna().astype(str))
asset_ids  = set(df_assets["pk_asset_id"].dropna().astype(str))

woo_sk     = set(df_woo["fk_required_skill_id"].dropna().astype(str))
woo_asset  = set(df_woo["fk_asset_id"].dropna().astype(str))

orphans_sk    = woo_sk - sk_ids
orphans_asset = woo_asset - asset_ids

print(f"fk_required_skill_id orphelins : {len(orphans_sk)}   {'✅' if len(orphans_sk)==0 else '❌ '+str(list(orphans_sk)[:3])}")
print(f"fk_asset_id orphelins          : {len(orphans_asset)} {'✅' if len(orphans_asset)==0 else '❌ '+str(list(orphans_asset)[:3])}")
print(f"\n{' Toutes les FK sont propres — prêt pour import SQLite' if len(orphans_sk)==0 and len(orphans_asset)==0 else '⚠️  Corriger les orphelins avant import'}")

 — fk_required_skill_id
   Valeurs uniques : [1, 2, 3]
   NULL restants   : 0

 _ fk_asset_id float → Int64
   Type actuel     : Int64
   NULL restants   : 113
   Exemple valeurs : [1, 1, 1, 1, 1]

 fk_assigned_technician_id float → Int64
   Type actuel     : Int64
   NULL restants   : 1096 (normal — pas encore assigné)

── Vérification après corrections 
fk_required_skill_id orphelins : 0   ✅
fk_asset_id orphelins          : 0 ✅

 Toutes les FK sont propres — prêt pour import SQLite


# Création de la base SQLite
### C'est une base de données locale — un seul fichier .db
### Zéro configuration, zéro serveur, fonctionne directement en Python
### Parfait pour prototyper l'algorithme de planification

### ÉTAPES :
###  — Créer le fichier .db et activer les FK
###  — Créer le schéma (structure des tables dans le bon ordre)
### — Importer les données depuis les DataFrames
### — Vérification finale

In [30]:
import os 
# Fermer toutes les connexions SQLite ouvertes avant de supprimer le fichier
try:
    conn.close()
    print(" Connexion précédente fermée")
except:
    print("ℹ  Aucune connexion ouverte")

 Connexion précédente fermée


In [ ]:
import sqlite3
import os

#  Connexion ──────────────────────────────────────────────
DB_PATH = "./besap_planner.db"

# Si la base existe déjà on la supprime pour repartir proprement
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(" Ancienne base supprimée")

conn = sqlite3.connect(DB_PATH)

# PRAGMA foreign_keys = ON → SQLite vérifie les FK à chaque insertion
# Sans ça SQLite accepte n'importe quelle valeur même si l'ID n'existe pas
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()
print(f" Connexion établie → {os.path.abspath(DB_PATH)}")

# Schéma SQL ─────────────────────────────────────────────
#
# ORDRE CRITIQUE — on crée d'abord les tables sans FK (référence)
# puis les tables qui dépendent d'elles
#
# 1. availability        → pas de FK
# 2. customer            → pas de FK
# 3. asset_criticality   → pas de FK
# 4. skills              → pas de FK
# 5. work_centers        → pas de FK
# 6. technicians         → dépend de skills, availability, work_centers
# 7. holidays            → dépend de technicians
# 8. assets              → dépend de work_centers, customer, asset_criticality
# 9. operations          → pas de FK
# 10. work_order_operations → dépend de operations, skills, technicians, assets

schema = """
CREATE TABLE IF NOT EXISTS availability (
    pk_working_hour_id          INTEGER PRIMARY KEY,
    status                      TEXT,
    working_hour_description    TEXT,
    start_hour                  TEXT,
    end_hour                    TEXT,
    pause_start                 TEXT,
    pause_end                   TEXT
);

CREATE TABLE IF NOT EXISTS customer (
    pk_customer_id              INTEGER PRIMARY KEY,
    status                      TEXT,
    customer_description        TEXT
);

CREATE TABLE IF NOT EXISTS asset_criticality (
    pk_criticality_name         TEXT PRIMARY KEY
);

CREATE TABLE IF NOT EXISTS skills (
    pk_skill_id                 INTEGER PRIMARY KEY,
    skill_description           TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS work_centers (
    pk_work_center_id           INTEGER PRIMARY KEY,
    work_center_status          TEXT,
    work_center_description     TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS technicians (
    pk_technician_id            INTEGER PRIMARY KEY,
    technician_status           TEXT,
    technician_first_name       TEXT NOT NULL,
    technician_last_name        TEXT NOT NULL,
    technician_full_name        TEXT,
    fk_skill_id                 INTEGER REFERENCES skills(pk_skill_id),
    fk_availability_id          INTEGER REFERENCES availability(pk_working_hour_id),
    address_street              TEXT,
    address_door                TEXT,
    address_post_code           TEXT,
    address_city                TEXT,
    fk_work_center_id           INTEGER REFERENCES work_centers(pk_work_center_id)
);

CREATE TABLE IF NOT EXISTS holidays (
    holiday_request_time        TEXT,
    status                      TEXT,
    fk_technician_id            INTEGER REFERENCES technicians(pk_technician_id),
    requested_for               TEXT,
    full_or_half_day            TEXT,
    reason                      TEXT
);

CREATE TABLE IF NOT EXISTS assets (
    pk_asset_id                 INTEGER PRIMARY KEY,
    asset_status                TEXT,
    asset_description           TEXT,
    address_street              TEXT,
    address_door                TEXT,
    address_post_code           TEXT,
    address_city                TEXT,
    asset_region                TEXT,
    fk_work_center_id           INTEGER REFERENCES work_centers(pk_work_center_id),
    fk_customer_id              INTEGER REFERENCES customer(pk_customer_id),
    fk_criticality              TEXT    REFERENCES asset_criticality(pk_criticality_name),
    created_at                  TEXT
);

CREATE TABLE IF NOT EXISTS operations (
    pk_operation_id             INTEGER PRIMARY KEY,
    operation_key               TEXT,
    operation_description       TEXT,
    duration                    TEXT,
    duration_unit               TEXT,
    predecessor_operation_key   TEXT,
    operation_type              TEXT,
    operation_subtype           TEXT,
    operation_order             INTEGER
);

CREATE TABLE IF NOT EXISTS work_order_operations (
    pk_woo_id                   INTEGER PRIMARY KEY,
    status                      TEXT,
    fk_operation_id             INTEGER REFERENCES operations(pk_operation_id),
    operation_key               TEXT,
    operation_description       TEXT,
    duration                    TEXT,
    duration_unit               TEXT,
    predecessor_operation_key   TEXT,
    fk_required_skill_id        INTEGER REFERENCES skills(pk_skill_id),
    required_skill_desc         TEXT,
    order_basic_start_date      TEXT,
    order_basic_end_date        TEXT,
    operation_scheduled_start   TEXT,
    operation_scheduled_end     TEXT,
    confirmed_work              TEXT,
    confirmed_work_unit         TEXT,
    fk_assigned_technician_id   INTEGER REFERENCES technicians(pk_technician_id),
    operation_type              TEXT,
    operation_subtype           TEXT,
    operation_order             INTEGER,
    fk_asset_id                 INTEGER REFERENCES assets(pk_asset_id),
    priority_score              INTEGER
);
"""

cur.executescript(schema)
conn.commit()
print(" Schéma SQL créé — 10 tables")

# Import des données 
#
# POURQUOI df.to_sql() ?
# C'est la façon la plus simple d'importer un DataFrame pandas
# directement dans SQLite sans écrire de INSERT manuellement
#
# if_exists="append" → ajoute les données sans écraser le schéma
# index=False        → n'importe pas l'index pandas comme colonne

print("\n── Import des données ")

import_order = [
    ("availability",            df_availability),
    ("customer",                df_customer),
    ("asset_criticality",       df_asset_crit),
    ("skills",                  df_skills),
    ("work_centers",            df_work_centers),
    ("technicians",             df_technicians),
    ("holidays",                df_holidays),
    ("assets",                  df_assets),
    ("operations",              df_operations),
    ("work_order_operations",   df_woo),
]

for table_name, df in import_order:
    try:
        df.to_sql(table_name, conn, if_exists="append", index=False)
        print(f" {table_name:<28} → {len(df)} lignes importées")
    except Exception as e:
        print(f" {table_name:<28} → ERREUR COMPLÈTE : {e}")
        # Afficher les colonnes du DataFrame vs le schéma SQLite
        print(f"\n   Colonnes DataFrame ({len(df.columns)}) :")
        for col in df.columns:
            print(f"      • {col}")
        # Afficher les colonnes de la table SQLite
        cols_sql = cur.execute(f"PRAGMA table_info({table_name})").fetchall()
        print(f"\n   Colonnes SQLite ({len(cols_sql)}) :")
        for col in cols_sql:
            print(f"      • {col[1]}")

#  Vérification finale 
#
# On compte les lignes dans chaque table de la BD
# et on vérifie que ça correspond aux DataFrames originaux

print("\n── Vérification finale ")
print(f"\n{'Table':<28} {'DataFrame':>12} {'SQLite':>10} {'OK ?':>8}")
print("-" * 62)

verification = [
    ("availability",            df_availability),
    ("customer",                df_customer),
    ("asset_criticality",       df_asset_crit),
    ("skills",                  df_skills),
    ("work_centers",            df_work_centers),
    ("technicians",             df_technicians),
    ("holidays",                df_holidays),
    ("assets",                  df_assets),
    ("operations",              df_operations),
    ("work_order_operations",   df_woo),
]

all_ok = True
for table_name, df in verification:
    count_sql = cur.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    count_df  = len(df)
    ok        = "" if count_sql == count_df else "❌"
    if count_sql != count_df:
        all_ok = False
    print(f"{table_name:<28} {count_df:>12} {count_sql:>10} {ok:>8}")

# Vérification priority_score
print("\n── Répartition priority_score dans SQLite ──────────────────")
scores = cur.execute("""
    SELECT priority_score, COUNT(*) as nb
    FROM work_order_operations
    GROUP BY priority_score
    ORDER BY priority_score DESC
""").fetchall()

labels = {5:"Breakdown", 4:"Meca", 3:"Elec", 2:"monthly", 1:"Low", 0:"Inconnu"}
for score, nb in scores:
    barre = "█" * (nb // 20)
    print(f"   Score {score} ({labels.get(score,'?'):<10}) → {nb:>4} opérations  {barre}")

conn.close()

if all_ok:
    print(f"\n Base SQLite créée avec succès !")
    print(f"   Fichier : {os.path.abspath(DB_PATH)}")
else:
    print(f"\n Certaines tables ont un nombre de lignes différent")
    print(f"   Vérifiez les erreurs ci-dessus")

 Ancienne base supprimée
 Connexion établie → c:\Maria\digitalPlanner\besap_planner.db
 Schéma SQL créé — 10 tables

── Import des données 
 availability                 → 2 lignes importées
 customer                     → 62 lignes importées
 asset_criticality            → 3 lignes importées
 skills                       → 4 lignes importées
 work_centers                 → 3 lignes importées
 technicians                  → 7 lignes importées
 holidays                     → 0 lignes importées
 assets                       → 145 lignes importées
 operations                   → 27 lignes importées
 work_order_operations        → 1096 lignes importées

── Vérification finale 

Table                           DataFrame     SQLite     OK ?
--------------------------------------------------------------
availability                            2          2         
customer                               62         62         
asset_criticality                       3          3         
skills

Scoring operation_type 